# Task 3

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
import os

import datavis
import models
import inference
import inference_analysis as IA
from HMM_inference import(
    perform_ramp_inference,
    plot_ramp_example,
    perform_step_inference,
    plot_step_example
)
import numpy as np
import matplotlib.pyplot as plt
from HMM_models import StepModelHMM, RampModelHMM, DiscreteRampHMM
import wrappers23new as g3
from itertools import product

from joblib import Parallel, delayed

# Set random seed for reproducibility
np.random.seed(42)

# Plotting settings
plt.style.use('seaborn-v0_8')
plt.rcParams['figure.figsize'] = [10, 6]
plt.rcParams['font.size'] = 12

## Parameter ranges

In [ ]:
# Parameter ranges
T = 100
K = 100

beta_range = [0, 4]
log_sigma_range = [np.log(0.04), np.log(4)]
r_range = [1, 10] #Step of 1
m_range = [0, T]
x0_range = [0, 0.5]

## Task 3.1

### Grid approximation:

Initial:
- fix value of $x_0$ to 0.2 (for both models) 
    - I.e. assumed to be known, and not inferred
- Construct 2D grid on each model's parameter space (for ranges given, using log sigma)
    - M grid points
    - Start with M = 30
- Simulate a dataset of $N_{trial}$ spike trains
    - Evaluate THAT model's log-likelihood on all grid points
        -  (by summing log-likelihood of all trials)
    - form normalised posterior

3.1.1
- Visualize approximate posterior (using imshow)
    - with superimposed point representing true parameters
- Repeat for different true parameters
    - within ranges
    - for trials fom 1 t 400
- document observations of sustematic dependence of posterior on number of trials
- plus other systematic dependencies of true parameters

3.1.2
- Evaluate posterior expectations as their estimates, as well as standard deviations
- Evaluate actual estimation error of different parameters and dataset sizes, or true parameters
    - average the error over 2-3 data sets when $N_{trial}$ is low
- Make plots
- Compare estimation errors with posterior uncertainties

3.1.3
- Repeat above, reducing M or number of explored cases, for when $x_0$ has to be inferred


## Task 3.2

### Model selection (grid based)

Initial:
- Repeat subtasks of 3.1 for the two "cross-cases"
    - I.e. compute posterior and posterior expectation of one models parameters when data is generated by the other model
    - Note changes
    - Document interesting observations and systematic behaviour
- Bayes factors
    - Take prior probabilities of models to be 0.5 each
    - Posterior model probabilities are proportional to marginal likelihoods
    - Model selection can be based on marginal likelihood ratio (MLR), aka bayes factor, (or logs of these)
    - Decision boundary of R > 1 or ln(R) > 0
    - To marginalise we integrate
    - This is exactly the normalising constant!
    - So yo have already calculated this in 3.1
    - make sure suitable differences in the parameters are used in the formula

Do model selection based on the MLR, for datasets of different sizes (number of trials) and quantify the two different error rates

3.2.1
- sample "true" parameters from that model's prior parameter distribution
    - (uniform for all? except log sigma)

3.2.2
- Repat with "mismatched" parameter priors
    - i.e. use different priors for parameter inference, with same sampling method
    - Use truncated Gaussian priors (independent) centred at different locations
        - m and sigma: centre in the middle of the ranges
        - r and beta: centre at the two extremes (4 total)
        - try different standard deviations (based on fractions of their total length to be consistent across parameters)
            - 2-3 different values for this
            - Use sensible values
- What happens to error rates?
    - try for 50, 100, 500, 1000, trial datasets
    - compare data generation distributions and priors

Note still using discretized parameter grid




## Task 3.1


In [ ]:
# Restate Parameter ranges
T = 100
K = 100

beta_range = [0, 4]
log_sigma_range = [np.log(0.04), np.log(4)]
r_range = [1, 10] #Step of 1
m_range = [0, T]
x0_range = [0, 0.5]
dt = 0.01

R_h     = 30.0        # max Poisson rate (Hz)


Ramp log likelihood

In [ ]:




def perform_ramp_inference_full(
    K: int,
    beta: float,
    sigma: float,
    dt: float,
    T: int,
    R_h: float,
    N: int,
    x0: np.ndarray = None,
):
    """
    Simulate N trials from the Ramp HMM, run HMM smoothing (or filtering), and return:
      - true_ramps    : shape (N, T+1), the ground-truth x_t trajectories
      - inferred_ramps: shape (N, T+1), E[x_t | spikes] for each trial
      - MAEs           : shape (N,), mean absolute error per trial

    Parameters
    ----------
    K       : number of discrete ramp levels (integer)
    beta    : drift parameter for the ramp-HMM (float)
    sigma   : diffusion parameter for the ramp-HMM (float)
    dt      : time-bin width (in seconds; float)
    T       : number of bins per trial (integer)
    R_h     : maximum (high) Poisson rate in Hz (float)
    N       : number of trials to simulate (integer)
    pi0     : length-K initial distribution; if None, assumes all mass on state 0 (np.ndarray of shape (K,))

    Returns
    -------
    true_ramps     : np.ndarray, shape (N, T+1)
    inferred_ramps : np.ndarray, shape (N, T+1)
    MAEs           : np.ndarray, shape (N,)
    """
    # 1) Build the Ramp HMM and transition matrix
    ramp_hmm = RampModelHMM(K=K, beta=beta, sigma=sigma, dt=dt)
    Ps = ramp_hmm.T       # (K × K) transition matrix

    # 2) Build pi0 if not provided
    if x0 is None:
        pi0 = np.zeros(K)
        pi0[0] = 1.0       # always start in state 0
    else:
        pi0 = initial_distribution(x0, sigma, dt)

    # 3) Precompute x_grid = [0, 1/(K-1), 2/(K-1), ..., 1]
    x_grid = np.arange(K, dtype=float) / float(K - 1)

    # 4) Storage for ground truth and inference results
    true_ramps     = np.zeros((N, T + 1), dtype=float)
    inferred_ramps = np.zeros((N, T + 1), dtype=float)
    inferred_ramps_filtered = np.zeros((N, T + 1), dtype=float)
    MAEs           = np.zeros(N, dtype=float)
    MAEs_filtered  = np.zeros(N, dtype=float)
    # 5) Loop over trials
    for i in range(N):
        # 5a) Simulate one trial: (states_i, x_vals_i, spikes_i)
        states_i, x_vals_i, spikes_i = ramp_hmm.simulate_spikes(
            n_steps=T,
            initial_state=0,
            R_h=R_h,
            dt=dt
        )
        true_ramps[i, :] = x_vals_i   # store ground‐truth

        # 5b) Build Poisson log‐likelihood matrix ll_i: shape (T+1, K)
        #    If latent state is s, rate = R_h * (s / (K - 1))
        rates = R_h * x_grid       # shape (K,)
        lambdas = rates * dt  # convert to expected counts per bin
        ll_i = inference.poisson_logpdf(
            counts=spikes_i,       # shape (T+1,)
            lambdas=lambdas,         # shape (K,)
            mask=None              # yields shape (T+1, K)
        )

        # 5c) Run forward–backward: smoothing
        post_probs_i, logZ_i = inference.hmm_expected_states(
            pi0=pi0,     # shape (K,)
            Ps=Ps,       # shape (K, K)
            ll=ll_i,     # shape (T+1, K)
            filter=False
        )
        # post_probs_i[t, s] = P(s_t = s | data)

        # 5d) Compute E[x_t | data] = sum_s [ x_grid[s] * post_probs_i[t, s] ]
        Exp_x_i = (post_probs_i * x_grid[None, :]).sum(axis=1)   # shape (T+1,)
        inferred_ramps[i, :] = Exp_x_i

        # 5e) Compute MAE for this trial
        MAEs[i] = np.mean(np.abs(Exp_x_i - x_vals_i))

    # added filtered part irrispective of input so can compare for some trials
        post_probs_i_filtered, logZ_i = inference.hmm_expected_states(
            pi0=pi0,     # shape (K,)
            Ps=Ps,       # shape (K, K)
            ll=ll_i,     # shape (T+1, K)
            filter=True
        )

        # 5d) Compute E[x_t | data] = sum_s [ x_grid[s] * post_probs_i[t, s] ]
        Exp_x_i_filtered = (post_probs_i_filtered * x_grid[None, :]).sum(axis=1)   # shape (T+1,)
        inferred_ramps_filtered[i, :] = Exp_x_i_filtered

        # 5e) Compute MAE for this trial
        MAEs_filtered[i] = np.mean(np.abs(Exp_x_i_filtered - x_vals_i))

    return true_ramps, inferred_ramps, inferred_ramps_filtered, MAEs, MAEs_filtered

In [ ]:
from scipy.stats import norm

beta = 2
sigma = 0.1
x0 = 0.2
dt = 0.01

def truncated_normal_pdf(x, mu, sigma, lower, upper):
    Z = norm.cdf(upper, loc=mu, scale=sigma) - norm.cdf(lower, loc=mu, scale=sigma)
    return norm.pdf(x, loc=mu, scale=sigma) / Z if Z > 0 else 0.0

def initial_distribution(x0, sigma, dt):
        mu = x0
        std = sigma * np.sqrt(dt)
        pi = np.array([truncated_normal_pdf(x, mu, std, 0, 1) for x in np.linspace(0, 1, K)])
        return pi / pi.sum()

ramp_hmm = RampModelHMM(K=K, beta=beta, sigma=sigma, dt=dt)
Ps = ramp_hmm.T       # (K × K) transition matrix
pi0 = initial_distribution(x0, sigma, dt)

states, x_vals, spikes = ramp_hmm.simulate_spikes(
    n_steps=T,
    initial_state=int(x0*K),
    R_h=R_h,
    dt=dt
)
x_grid = np.arange(K, dtype=float) / float(K - 1)
# 5b) Build Poisson log‐likelihood matrix ll_i: shape (T+1, K)
        #    If latent state is s, rate = R_h * (s / (K - 1))
rates = R_h * x_grid       # shape (K,)
lambdas = rates * dt  # convert to expected counts per bin
ll = inference.poisson_logpdf(
    counts=spikes,       # shape (T+1,)
    lambdas=lambdas,         # shape (K,)
    mask=None              # yields shape (T+1, K)
)

normalizer = inference.hmm_normalizer(pi0, Ps, ll)

print(normalizer)


In [ ]:
M = 30  # number of grid points
x0 = 0.2  # fixed initial state

n_jobds = -1 # use all cores

# Parameter ranges
beta_vals = np.linspace(0, 4.0, M)
log_sigma_vals = np.linspace(np.log(0.04), np.log(4.0), M)
sigma_vals = np.exp(log_sigma_vals)  # log scale for sigma

# Create grid
param_grid = [(beta, sigma) for beta in beta_vals for sigma in sigma_vals]



def simulate_spike_train_dataset(beta, sigma, dt=dt, K=K):
        ramp_hmm = RampModelHMM(K=K, beta=beta, sigma=sigma, dt=dt)
        _, _, _, mae, mae_f = perform_ramp_inference(
            K=K, beta=beta, sigma=sigma, dt=dt, T=T, R_h=R_h, N=N
        )
        return np.mean(mae), np.mean(mae_f)

results = Parallel(n_jobs=n_jobs)(
        delayed(compute_error)(beta, sigma) for beta, sigma in param_grid
    )

ramp_hmm = RampModelHMM(K=K, beta=beta, sigma=sigma, dt=dt)

N_trials = 50
ramp_trials = [simulate_ramp_spike_train(beta=0.4, sigma=0.1, x0=x0, T=T, dt=dt) for _ in range(N_trials)]
